<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_05_02_random_forest_one2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_05_02 - ONE2ONE - Random Forest**

## **Introducción**

**Random Forest** es un modelo de ensamble basado en múltiples **árboles de decisión**.
Cada árbol aprende reglas no lineales sobre distintas muestras del dataset, y la predicción final es el **promedio** de todos los árboles.

Formalmente:

$$
\hat{y} = \frac{1}{N} \sum_{i=1}^{N} T_i(x)
$$

donde $T_i$ es un árbol entrenado sobre una muestra aleatoria.

---

**Cómo funciona**

* Cada árbol:

  * se entrena con **bootstrap sampling** (muestras con reemplazo)
  * usa un subconjunto aleatorio de features en cada split
* Esto introduce:

  * **diversidad entre árboles**
  * reducción de varianza

---

**Propiedades clave**

* Captura **relaciones no lineales**
* Maneja bien:

  * interacciones entre variables
  * features correlacionadas
* No requiere:

  * escalado
  * supuestos de distribución

---

**Regularización implícita**

Se controla mediante:

* `max_depth`
* `min_samples_leaf`
* `max_features`
* `n_estimators`

El modelo reduce overfitting promediando múltiples árboles.

---

**Por qué usarlo en este proyecto**

* Ridge falló → no hay señal lineal clara
* Random Forest permite detectar:

  * **no linealidades**
  * **threshold effects**
  * **interacciones entre indicadores técnicos**

Es el siguiente paso natural como baseline **no lineal**.

---

**Rol en el pipeline**

* Modelo más potente que Ridge
* Aún interpretable (feature importance)
* Buen benchmark antes de modelos más complejos (XGBoost, DL)


# **Bloque común**

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de variables X e y, y scalers**

In [3]:
from pathlib import Path
import os
import pandas as pd
import joblib


XY_DELTA_DIR = DRIVE_DIR / Path(
    os.environ.get("XY_DELTA_DIR", "data/splits/")
)

XY_DELTA_DIR_SCALED = DRIVE_DIR / Path(
    os.environ.get("XY_DELTA_DIR_SCALED", "data/scaled/")
)

SCALERS_DIR = DRIVE_DIR / Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

TARGETS = ["delta_60", "delta_90"]
SPLITS = ["train", "valid", "test"]


def load_mnq_tabular_split(
    target: str,
    split: str,
    scaled: bool = False,
    return_scaler: bool = False,
):
    if target not in TARGETS:
        raise ValueError(f"target inválido: {target}. Esperados: {TARGETS}")

    if split not in SPLITS:
        raise ValueError(f"split inválido: {split}. Esperados: {SPLITS}")

    x_path = (
        XY_DELTA_DIR_SCALED / f"mnq_{target}_X_{split}_scaled.parquet"
        if scaled
        else XY_DELTA_DIR / f"mnq_{target}_X_{split}.parquet"
    )
    y_path = XY_DELTA_DIR / f"mnq_{target}_y_{split}.parquet"

    if not x_path.exists():
        raise FileNotFoundError(f"No existe X: {x_path}")
    if not y_path.exists():
        raise FileNotFoundError(f"No existe y: {y_path}")

    X = pd.read_parquet(x_path)
    y = pd.read_parquet(y_path)

    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    if not return_scaler:
        return X, y

    scaler = None
    if scaled:
        scaler_path = SCALERS_DIR / f"scaler_{target}.pkl"
        if scaler_path.exists():
            scaler = joblib.load(scaler_path)

    return X, y, scaler

In [4]:
#Sin escalado
SIN_ESCALADO = '''
X_train, y_train = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=False,
)
'''

In [5]:
#Escalado
ESCALADO = '''
X_train, y_train = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=True,
)
'''


ESCALADO_ESCALADOR = '''
X_train, y_train, scaler = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=True,
    load_scaler=True,
)
'''

In [6]:
import pandas as pd
import numpy as np


def validate_tabular_dataset(
    X: pd.DataFrame,
    y: pd.Series | pd.DataFrame,
    *,
    name: str = "",
    check_index_alignment: bool = True,
    check_sorted: bool = True,
    date_col: str | None = None,
    verbose: bool = True,
):
    """
    Valida consistencia de un dataset tabular (X, y).

    Checks:
    - shapes
    - NaNs / inf
    - alineación de índices
    - orden temporal (opcional)
    - duplicados

    Retorna
    -------
    dict con flags de validación
    """

    report = {}

    # -------- Convertir y --------
    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    # -------- Shapes --------
    report["n_samples_X"] = X.shape[0]
    report["n_samples_y"] = y.shape[0]
    report["n_features"] = X.shape[1]
    report["shape_match"] = X.shape[0] == y.shape[0]

    # -------- NaNs / inf --------
    report["X_has_nan"] = X.isna().any().any()
    report["y_has_nan"] = y.isna().any()

    report["X_has_inf"] = np.isinf(X.select_dtypes(include=[np.number])).any().any()
    report["y_has_inf"] = np.isinf(y).any()

    # -------- Índices --------
    if check_index_alignment:
        report["index_equal"] = X.index.equals(y.index)
    else:
        report["index_equal"] = None

    # -------- Orden temporal --------
    if check_sorted:
        if date_col and date_col in X.columns:
            report["sorted_by_date"] = X[date_col].is_monotonic_increasing
        else:
            report["sorted_by_index"] = X.index.is_monotonic_increasing
    else:
        report["sorted"] = None

    # -------- Duplicados --------
    report["duplicate_index"] = X.index.duplicated().any()

    # -------- Print --------
    if verbose:
        print(f"\n=== VALIDATION: {name} ===")
        for k, v in report.items():
            print(f"{k}: {v}")

        if not report["shape_match"]:
            print("⚠️ ERROR: X e y no tienen mismo número de filas")

        if report["X_has_nan"] or report["y_has_nan"]:
            print("⚠️ WARNING: Hay NaNs")

        if report["X_has_inf"] or report["y_has_inf"]:
            print("⚠️ WARNING: Hay valores infinitos")

        if check_index_alignment and not report["index_equal"]:
            print("⚠️ WARNING: Índices no alineados")

    return report

## **4. Reproducibilidad**

In [7]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
from pathlib import Path
import os
import sys
import importlib

DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

from metrics.one2one_metrics import evaluate_regression_predictions, print_metrics

print("OK - imports metrics.*")

import metrics.one2one_metrics as m

#print(m.__doc__)
#print(m.evaluate_regression_predictions.__doc__)


OK - imports metrics.*


## **6. Métricas Machine Learning**

In [9]:
import pandas as pd


def evaluate_model_on_split(
    model,
    X,
    y,
    *,
    split_name: str,
    model_name: str,
    target_name: str,
):
    """
    Evalúa un modelo sobre un split dado y devuelve:
    - y_pred
    - dict de métricas
    """
    y_pred = model.predict(X)

    metrics = evaluate_regression_predictions(
        y_true=y,
        y_pred=y_pred,
        split_name=split_name,
        model_name=model_name,
        target_name=target_name,
    )

    return y_pred, metrics


def metrics_to_df(metrics: dict) -> pd.DataFrame:
    """
    Convierte un dict de métricas en una fila de DataFrame.
    Versión final sin redundancias (ni window_size ni horizon).
    """
    row = {
        "model": metrics.get("model"),
        "split": metrics.get("split"),
        "target": metrics.get("target"),
        "n_samples": metrics.get("n_samples"),
        "mae": metrics.get("mae"),
        "rmse": metrics.get("rmse"),
        "r2": metrics.get("r2"),
        "directional_accuracy": metrics.get("directional_accuracy"),
    }

    return pd.DataFrame([row])

def evaluate_model_on_bundle(
    model,
    bundle: dict,
    *,
    model_name: str,
    target_name: str,
    window_size: int | None = None,
    horizon: int | None = None,
    splits: tuple[str, ...] = ("valid", "test"),
):
    """
    Evalúa un modelo en varios splits de un bundle.

    Estructura esperada de bundle:
    bundle = {
        "train": {"X": ..., "y": ...},
        "valid": {"X": ..., "y": ...},
        "test":  {"X": ..., "y": ...},
    }

    Retorna
    -------
    predictions : dict
        Predicciones por split.
    metrics_dict : dict
        Métricas por split.
    metrics_df : pd.DataFrame
        Tabla consolidada.
    """
    predictions = {}
    metrics_dict = {}
    frames = []

    for split in splits:
        X = bundle[split]["X"]
        y = bundle[split]["y"]

        y_pred, metrics = evaluate_model_on_split(
            model=model,
            X=X,
            y=y,
            split_name=split,
            model_name=model_name,
            target_name=target_name,
        )

        predictions[split] = y_pred
        metrics_dict[split] = metrics
        frames.append(
            metrics_to_df(
                metrics,
                window_size=window_size,
                horizon=horizon,
            )
        )

    metrics_df = pd.concat(frames, ignore_index=True)

    return predictions, metrics_dict, metrics_df

## **7. Gestión de dataset de métricas**

In [10]:
def load_one2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/one2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"one2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [11]:
from pathlib import Path
import pandas as pd

def save_one2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/one2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"one2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

# **DEFINICIÓN DE MODELO**

## **8. Definición del modelo — placeholder**

### **8.1. Modelo Ridge Regression (one2one)**

### **8.2. Carga de X e y**

In [12]:
TARGETS = ["delta_60", "delta_90"]

data = {}

for target in TARGETS:
    print(f"\n==============================")
    print(f"CARGANDO DATASET: {target}")
    print(f"==============================")

    # -------- TRAIN --------
    X_train, y_train = load_mnq_tabular_split(
        target=target,
        split="train",
        scaled=False,
    )

    validate_tabular_dataset(
        X_train,
        y_train,
        name=f"train_{target}",
    )

    # -------- VALID --------
    X_valid, y_valid = load_mnq_tabular_split(
        target=target,
        split="valid",
        scaled=False,
    )

    validate_tabular_dataset(
        X_valid,
        y_valid,
        name=f"valid_{target}",
    )

    # -------- TEST --------
    X_test, y_test = load_mnq_tabular_split(
        target=target,
        split="test",
        scaled=False,
    )

    validate_tabular_dataset(
        X_test,
        y_test,
        name=f"test_{target}",
    )

    # -------- Guardar --------
    data[target] = {
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }


CARGANDO DATASET: delta_60

=== VALIDATION: train_delta_60 ===
n_samples_X: 54360
n_samples_y: 54360
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

=== VALIDATION: valid_delta_60 ===
n_samples_X: 11640
n_samples_y: 11640
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

=== VALIDATION: test_delta_60 ===
n_samples_X: 11700
n_samples_y: 11700
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

CARGANDO DATASET: delta_90

=== VALIDATION: train_delta_90 ===
n_samples_X: 54360
n_samples_y: 54360
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: Fal

In [13]:
X_train_delta_60 = data["delta_60"]["train"]["X"]
y_train_delta_60 = data["delta_60"]["train"]["y"]

X_valid_delta_60 = data["delta_60"]["valid"]["X"]
y_valid_delta_60 = data["delta_60"]["valid"]["y"]

X_test_delta_60  = data["delta_60"]["test"]["X"]
y_test_delta_60  = data["delta_60"]["test"]["y"]

X_train_delta_90 = data["delta_90"]["train"]["X"]
y_train_delta_90 = data["delta_90"]["train"]["y"]

X_valid_delta_90 = data["delta_90"]["valid"]["X"]
y_valid_delta_90 = data["delta_90"]["valid"]["y"]

X_test_delta_90  = data["delta_90"]["test"]["X"]
y_test_delta_90  = data["delta_90"]["test"]["y"]

### **8.3. Entrenamiento**

In [14]:
from sklearn.ensemble import RandomForestRegressor


def train_evaluate_random_forest_one2one(
    *,
    target_name: str,
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    n_estimators: int = 200,
    max_depth: int | None = None,
    min_samples_split: int = 2,
    min_samples_leaf: int = 1,
    max_features: str | int | float | None = "sqrt",
    random_state: int = 42,
    n_jobs: int = -1,
):
    """
    Entrena y evalúa un Random Forest Regressor one-to-one para un target dado.

    Retorna
    -------
    results : dict
        Contiene modelo, predicciones y métricas por split.
    """
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=random_state,
        n_jobs=n_jobs,
    )

    # -------------------------
    # Entrenamiento
    # -------------------------
    model.fit(X_train, y_train)

    # -------------------------
    # Predicciones
    # -------------------------
    y_pred_valid = model.predict(X_valid)
    y_pred_test = model.predict(X_test)

    # -------------------------
    # Métricas
    # -------------------------
    metrics_valid = evaluate_regression_predictions(
        y_true=y_valid,
        y_pred=y_pred_valid,
        split_name="valid",
        model_name="random_forest",
        target_name=target_name,
    )

    metrics_test = evaluate_regression_predictions(
        y_true=y_test,
        y_pred=y_pred_test,
        split_name="test",
        model_name="random_forest",
        target_name=target_name,
    )

    results = {
        "model": model,
        "target": target_name,
        "predictions": {
            "valid": y_pred_valid,
            "test": y_pred_test,
        },
        "metrics": {
            "valid": metrics_valid,
            "test": metrics_test,
        },
    }

    return results

In [15]:
rf_delta_60 = train_evaluate_random_forest_one2one(
    target_name="delta_60",
    X_train=X_train_delta_60,
    y_train=y_train_delta_60,
    X_valid=X_valid_delta_60,
    y_valid=y_valid_delta_60,
    X_test=X_test_delta_60,
    y_test=y_test_delta_60,
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

print_metrics(rf_delta_60["metrics"]["valid"])
print_metrics(rf_delta_60["metrics"]["test"])

=== Regression Metrics ===
model: random_forest
target: delta_60
split: valid
n_samples: 11640
mae: 46.730486
rmse: 60.683050
r2: -0.086106
directional_accuracy: 0.489691
=== Regression Metrics ===
model: random_forest
target: delta_60
split: test
n_samples: 11700
mae: 78.649634
rmse: 107.603063
r2: -0.022491
directional_accuracy: 0.504615


In [16]:
rf_delta_90 = train_evaluate_random_forest_one2one(
    target_name="delta_90",
    X_train=X_train_delta_90,
    y_train=y_train_delta_90,
    X_valid=X_valid_delta_90,
    y_valid=y_valid_delta_90,
    X_test=X_test_delta_90,
    y_test=y_test_delta_90,
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

print_metrics(rf_delta_90["metrics"]["valid"])
print_metrics(rf_delta_90["metrics"]["test"])

=== Regression Metrics ===
model: random_forest
target: delta_90
split: valid
n_samples: 11640
mae: 62.378086
rmse: 80.969334
r2: -0.082563
directional_accuracy: 0.492268
=== Regression Metrics ===
model: random_forest
target: delta_90
split: test
n_samples: 11700
mae: 102.905534
rmse: 134.987579
r2: -0.013262
directional_accuracy: 0.495556


### **8.4. Guardado de datasets de métricas**

In [17]:
def build_and_save_one2one_metrics(
    *,
    results_delta_60: dict,
    results_delta_90: dict,
    model_name: str,
):
    """
    Construye y guarda métricas one2one para ambos targets.

    Parámetros
    ----------
    results_delta_60 : dict
    results_delta_90 : dict
    model_name : str
        Nombre del modelo (ej: 'ridge', 'random_forest')
    """

    # =========================================================
    # 1. DELTA 60
    # =========================================================
    df_valid_60 = metrics_to_df(
        results_delta_60["metrics"]["valid"]
    )

    df_test_60 = metrics_to_df(
        results_delta_60["metrics"]["test"]
    )

    df_60 = pd.concat([df_valid_60, df_test_60], ignore_index=True)

    # =========================================================
    # 2. DELTA 90
    # =========================================================
    df_valid_90 = metrics_to_df(
        results_delta_90["metrics"]["valid"]
    )

    df_test_90 = metrics_to_df(
        results_delta_90["metrics"]["test"]
    )

    df_90 = pd.concat([df_valid_90, df_test_90], ignore_index=True)

    # =========================================================
    # 3. CONSOLIDACIÓN
    # =========================================================
    df_all = pd.concat([df_60, df_90], ignore_index=True)

    # =========================================================
    # 4. GUARDADO
    # =========================================================
    save_one2one_metrics(
        df_all,
        name=f"{model_name}_all",
    )

    return df_all

In [18]:
df_rf = build_and_save_one2one_metrics(
    results_delta_60=rf_delta_60,
    results_delta_90=rf_delta_90,
    model_name="random_forest",
)

df_rf

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/one2one_metrics/one2one_random_forest_all_metrics.parquet


,model,split,target,n_samples,mae,rmse,r2,directional_accuracy
0,random_forest,valid,delta_60,11640,46.730486,60.683050,-0.086106,0.489691
1,random_forest,test,delta_60,11700,78.649634,107.603063,-0.022491,0.504615
2,random_forest,valid,delta_90,11640,62.378086,80.969334,-0.082563,0.492268
3,random_forest,test,delta_90,11700,102.905534,134.987579,-0.013262,0.495556


### **PREMARKET — Random Forest**

| model         | split | target   | n_samples | mae        | rmse       | r2        | directional_accuracy |
| ------------- | ----- | -------- | --------- | ---------- | ---------- | --------- | -------------------- |
| random_forest | valid | delta_60 | 11640     | 46.730486  | 60.683050  | -0.086106 | 0.489691             |
| random_forest | test  | delta_60 | 11700     | 78.649634  | 107.603063 | -0.022491 | 0.504615             |
| random_forest | valid | delta_90 | 11640     | 62.378086  | 80.969334  | -0.082563 | 0.492268             |
| random_forest | test  | delta_90 | 11700     | 102.905534 | 134.987579 | -0.013262 | 0.495556             |

---

### **OPENING — Random Forest**

| model         | split | target   | n_samples | mae       | rmse       | r2        | directional_accuracy |
| ------------- | ----- | -------- | --------- | --------- | ---------- | --------- | -------------------- |
| random_forest | valid | delta_60 | 11640     | 53.294412 | 69.903988  | -0.031515 | 0.501203             |
| random_forest | test  | delta_60 | 11700     | 76.455132 | 103.077275 | -0.009094 | 0.502650             |
| random_forest | valid | delta_90 | 11640     | 60.551008 | 81.397496  | -0.029146 | 0.502835             |
| random_forest | test  | delta_90 | 11700     | 86.381954 | 114.767816 | -0.022735 | 0.505897             |

---

### **REGULAR — Random Forest**

| model         | split | target   | n_samples | mae       | rmse       | r2        | directional_accuracy |
| ------------- | ----- | -------- | --------- | --------- | ---------- | --------- | -------------------- |
| random_forest | valid | delta_60 | 46754     | 36.182977 | 49.579952  | -0.025241 | 0.511208             |
| random_forest | test  | delta_60 | 46995     | 55.254131 | 89.449069  | -0.057181 | 0.497989             |
| random_forest | valid | delta_90 | 46754     | 43.863645 | 61.260822  | -0.028486 | 0.509304             |
| random_forest | test  | delta_90 | 46995     | 68.087058 | 112.542183 | -0.043309 | 0.494329             |

